# Linear Regression — insurance.csv
Predict medical insurance **charges** from age, sex, bmi, children, smoker, region.

## 1. Data Loading

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

df = pd.read_csv("insurance.csv")
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [2]:
df.shape

(1338, 7)

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   str    
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   str    
 5   region    1338 non-null   str    
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 73.3 KB


## 2. Preprocessing

In [4]:
print("Missing values:\n", df.isna().sum())
df = df.dropna()

Missing values:
 age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64


In [5]:
# One-hot encode categorical columns: sex, smoker, region
categorical_cols = ["sex", "smoker", "region"]
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
df_encoded.head()

,age,bmi,children,charges,sex_male,smoker_yes,region_northwest,region_southeast,region_southwest
0,19,27.900,0,16884.92400,False,True,False,False,True
1,18,33.770,1,1725.55230,True,False,False,True,False
2,28,33.000,3,4449.46200,True,False,False,True,False
3,33,22.705,0,21984.47061,True,False,True,False,False
4,32,28.880,0,3866.85520,True,False,True,False,False


In [6]:
X = df_encoded.drop(columns=["charges"])
y = df_encoded["charges"]
feature_columns = X.columns.tolist()
feature_columns

['age',
 'bmi',
 'children',
 'sex_male',
 'smoker_yes',
 'region_northwest',
 'region_southeast',
 'region_southwest']

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Train shape:", X_train.shape, " Test shape:", X_test.shape)

Train shape: (1070, 8)  Test shape: (268, 8)


## 3. Model Training

In [8]:
model = LinearRegression()
model.fit(X_train, y_train)
print("Intercept:", model.intercept_)

Intercept: -11931.219050326688


## 4. Model Evaluation — MAE & R² Score

In [9]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"R2 Score: {r2:.4f}")

Mean Absolute Error (MAE): 4181.1945
R2 Score: 0.7836


In [10]:
pd.DataFrame({"Actual": y_test, "Predicted": y_pred}).head(10)

,Actual,Predicted
764,9095.06825,8969.550274
887,5272.17580,7068.747443
890,29330.98315,36858.410912
1293,9301.89355,9454.678501
259,33750.29180,26973.173457
1312,4536.25900,10864.113164
899,2117.33885,170.280841
752,14210.53595,16903.450287
1286,3732.62510,1092.430936
707,10264.44210,11218.343184


## 5. Predictive Function

In [11]:
def predict_charges(age, sex, bmi, children, smoker, region):
    """
    Predict insurance charges for a new person.
    sex: 'male' or 'female'
    smoker: 'yes' or 'no'
    region: 'northeast', 'northwest', 'southeast', or 'southwest'
    """
    raw_input = pd.DataFrame([{
        "age": age, "sex": sex, "bmi": bmi, "children": children,
        "smoker": smoker, "region": region
    }])
    raw_encoded = pd.get_dummies(raw_input, columns=categorical_cols, drop_first=True)
    raw_encoded = raw_encoded.reindex(columns=feature_columns, fill_value=0)
    prediction = model.predict(raw_encoded)[0]
    return prediction

## 6. Prediction (example)

In [12]:
predicted_charge = predict_charges(
    age=30, sex="male", bmi=25.0, children=1, smoker="no", region="southeast"
)
print(f"Predicted Insurance Charges: {predicted_charge:.2f}")

Predicted Insurance Charges: 4630.64
